A pedagogical walkthrough using the Kaggle Credit Card Fraud dataset (~0.17% positive rate).

**Plan**

1. Load and inspect the data.
2. Stratified train/val split.
3. Baseline: logistic regression on the full imbalanced training set.
4. Comparison: logistic regression on a 1:1 downsampled training set.
5. Compare metrics (Average Precision, ROC-AUC, precision/recall/F1 at threshold, confusion matrix).
6. Compare predicted-score distributions on the validation set.

## 1. Load data

In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from dotenv import load_dotenv

# Registers and activates the shared `dvq` Plotly template.
import dvq_theme

# Load credentials from the repo-root .env, then map KAGGLE_API_TOKEN (KGAT_...) into KAGGLE_KEY for kagglehub.
load_dotenv(Path.cwd().parent.parent / ".env")
if os.environ.get("KAGGLE_API_TOKEN") and not os.environ.get("KAGGLE_KEY"):
    os.environ["KAGGLE_KEY"] = os.environ["KAGGLE_API_TOKEN"]

RANDOM_STATE = 42
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

In [2]:
import kagglehub

dataset_path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")
csv_path = Path(dataset_path) / "creditcard.csv"
print(csv_path)

df = pd.read_csv(csv_path)
print(df.shape)
df['Class'].value_counts(normalize=True)

/Users/dvq/.cache/kagglehub/datasets/mlg-ulb/creditcardfraud/versions/3/creditcard.csv


(284807, 31)


Class
0    0.998273
1    0.001727
Name: proportion, dtype: float64

## 2. Train/val split (stratified)

In [3]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["Class"])
y = df["Class"]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print(f"train: {len(X_train):>7,}   positives: {int(y_train.sum()):>5,}  ({y_train.mean()*100:.3f}%)")
print(f"val:   {len(X_val):>7,}   positives: {int(y_val.sum()):>5,}  ({y_val.mean()*100:.3f}%)")

train: 227,845   positives:   394  (0.173%)
val:    56,962   positives:    98  (0.172%)


## 3. Baseline: full imbalanced training set

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

baseline = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])
baseline.fit(X_train, y_train)

scores_baseline = baseline.predict_proba(X_val)[:, 1]

## 4. Comparison: 1:1 downsampled training set

In [5]:
pos_idx = y_train[y_train == 1].index
neg_idx = y_train[y_train == 0].index
neg_sampled = np.random.RandomState(RANDOM_STATE).choice(neg_idx, size=len(pos_idx), replace=False)
ds_idx = np.concatenate([pos_idx, neg_sampled])

X_train_ds = X_train.loc[ds_idx]
y_train_ds = y_train.loc[ds_idx]
print(f"downsampled train: {len(X_train_ds):,}  (positives: {int(y_train_ds.sum()):,})")

downsampled = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])
downsampled.fit(X_train_ds, y_train_ds)

scores_downsampled = downsampled.predict_proba(X_val)[:, 1]

downsampled train: 788  (positives: 394)


## 5. Metric comparison

In [6]:
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

def metrics_row(name, y_true, scores, threshold=0.5):
    y_pred = (scores >= threshold).astype(int)
    return {
        "model": name,
        "AP": average_precision_score(y_true, scores),
        "ROC-AUC": roc_auc_score(y_true, scores),
        f"precision@{threshold}": precision_score(y_true, y_pred, zero_division=0),
        f"recall@{threshold}": recall_score(y_true, y_pred, zero_division=0),
        f"f1@{threshold}": f1_score(y_true, y_pred, zero_division=0),
    }

metrics_df = pd.DataFrame([
    metrics_row("imbalanced (full)", y_val, scores_baseline),
    metrics_row("downsampled 1:1", y_val, scores_downsampled),
])
print(metrics_df.round(4).to_string(index=False))

            model     AP  ROC-AUC  precision@0.5  recall@0.5  f1@0.5
imbalanced (full) 0.7414   0.9605         0.8267      0.6327  0.7168
  downsampled 1:1 0.4578   0.9752         0.0516      0.9184  0.0977


In [7]:
for name, scores in [("imbalanced (full)", scores_baseline), ("downsampled 1:1", scores_downsampled)]:
    y_pred = (scores >= 0.5).astype(int)
    cm = confusion_matrix(y_val, y_pred)
    print(f"\n{name} @ threshold=0.5")
    print(pd.DataFrame(cm, index=["actual 0", "actual 1"], columns=["pred 0", "pred 1"]))


imbalanced (full) @ threshold=0.5
          pred 0  pred 1
actual 0   56851      13
actual 1      36      62

downsampled 1:1 @ threshold=0.5
          pred 0  pred 1
actual 0   55209    1655
actual 1       8      90


## 6. Score distributions on the validation set

In [8]:
#| column: page
fig = make_subplots(rows=1, cols=2, subplot_titles=("imbalanced (full)", "downsampled 1:1"), shared_yaxes=True)

panels = [(scores_baseline, 1), (scores_downsampled, 2)]
class_colors = {0: dvq_theme.ACCENT, 1: "#F58518"}

for scores, col in panels:
    for label in (0, 1):
        fig.add_trace(
            go.Histogram(
                x=scores[y_val == label],
                xbins=dict(start=0, end=1, size=0.02),
                histnorm="probability density",
                name=f"class {label}",
                legendgroup=f"class {label}",
                showlegend=(col == 1),
                marker=dict(color=class_colors[label], line=dict(width=0)),
                opacity=0.65,
            ),
            row=1,
            col=col,
        )

fig.update_layout(
    barmode="overlay",
    height=380,
    autosize=True,
    title="Predicted P(class=1) on validation set",
    legend=dict(orientation="h", y=-0.18, x=0.5, xanchor="center"),
)
fig.update_xaxes(title_text="predicted P(class=1)", range=[0, 1])
fig.update_yaxes(type="log", title_text="density (log)", col=1)
from IPython.display import HTML
HTML(fig.to_html(
    include_plotlyjs="cdn",
    full_html=False,
    div_id="fig-scores",
    config={"responsive": True},
    default_width="100%",
    default_height="380px",
))

## Next

Decide where to go from here based on what the comparison reveals — calibration check, threshold tuning, class weights, SMOTE, gradient boosting baseline, etc.